# visual-tcav — End-to-End Demo

This notebook demonstrates the full capabilities of the `visual-tcav` package
using ResNet50 pretrained on ImageNet and concept images from the
[DTD (Describable Textures Dataset)](https://www.robots.ox.ac.uk/~vgg/data/dtd/).

**What you will see:**
- How to inspect available layers before configuring the explainer
- Two ways to load a model (string or nn.Module)
- Two instantiation styles (constructor vs step-by-step)
- How `explain()` works without an explicit `predict()` call
- Concept maps across multiple CNN layers
- How to plug in a custom CAV computation function
- Global attribution statistics with confidence intervals
- How to interpret concept maps and attribution scores

**Reference paper:**
De Santis et al., *Visual-TCAV: Concept-based Attribution and Saliency Maps
for Post-hoc Explainability in Image Classification*, 2025.
[arXiv:2411.05698](https://arxiv.org/abs/2411.05698)

---

## 0. Installation

```bash
pip install visual-tcav
```

For the Text-to-Concept extension (requires CLIP):
```bash
pip install visual-tcav[text-to-concept]
```

## 1. Setup

Import the package and configure paths.

All paths are fully configurable — organize your files however you prefer.
The only convention is that random images must be in their own folder.

In [ ]:
import os
import torch
import torchvision.models as models
import matplotlib.pyplot as plt

from visual_tcav import (
    available_layers,
    LocalVisualTCAV,
    GlobalVisualTCAV,
    Cav,
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# All paths are fully configurable — set them to wherever your files live
CONCEPT_DIR     = "./data/concept_images"
RANDOM_DIR      = "./data/concept_images/random"
TEST_IMAGES_DIR = "./data/test_images"
CACHE_DIR       = "./data/.cache"

# Verify all required paths exist
for name, path in [
    ("concept_images", CONCEPT_DIR),
    ("test_images",    TEST_IMAGES_DIR),
    ("random",         RANDOM_DIR),
]:
    status = "OK" if os.path.exists(path) else "NOT FOUND"
    print(f"  [{status}] {name}: {path}")

## 2. Inspect available layers

Before creating the explainer, use `available_layers()` to see which
CNN layers can be analyzed. Use the names shown here in `layer_names`.

Deeper layers (closer to the output) capture higher-level concepts
such as textures and object parts. For ResNet50, `layer4` is a good
starting point.

In [ ]:
# Inspect layers before instantiating the explainer
# Accepts a model name string or an nn.Module directly
available_layers("resnet50")

In [ ]:
# Also works with an nn.Module
my_resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
available_layers(my_resnet)

---

## 3. LocalVisualTCAV — Option A: model name string

The simplest usage. Pass the model name as a string and the package
loads it automatically with default ImageNet weights.

Supported models: `resnet18`, `resnet50`, `resnet101`, `vgg16`, `vgg19`.

In [ ]:
tcav_local = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped", "dotted", "zigzagged"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    n_classes=3,
    cache_dir=CACHE_DIR,
)

### 3.1 Predict

`predict()` runs the image through the model and returns the top predicted classes.

In [ ]:
predictions = tcav_local.predict()
predictions.info(num_of_classes=5)

### 3.2 Explain

`explain()` runs the full Visual-TCAV pipeline.
It calls `predict()` automatically if not already done.

In [ ]:
tcav_local.explain(cache_cav=True, cache_random=True)

### 3.3 Visualize

In [ ]:
tcav_local.plot(figsize=(14, 10))

---

## 4. LocalVisualTCAV — Option B: nn.Module

Pass any PyTorch model directly. Use this for fine-tuned models,
custom architectures, or non-standard weights.
Set `model_name` to enable auto-loading of labels for known models.

In [ ]:
tcav_module = LocalVisualTCAV(
    model=my_resnet,
    model_name="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "honeycomb.jpg"),
    concept_names=["honeycombed", "waffled", "chequered"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer3", "layer4"],
    cache_dir=CACHE_DIR,
)

# explain() calls predict() automatically — no need to call it explicitly
tcav_module.explain(cache_cav=True, cache_random=True)
tcav_module.plot(figsize=(18, 10))

---

## 5. Concept maps across multiple layers

Analyzing the same concept at different layers shows how detection evolves with depth.

In [ ]:
tcav_multilayer = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer2", "layer3", "layer4"],
    cache_dir=CACHE_DIR,
)

tcav_multilayer.explain(cache_cav=True, cache_random=True)
tcav_multilayer.plot(figsize=(16, 5))

---

## 6. Custom CAV computation function

Replace the default centroid difference method with any custom function
via the `cav_fn` parameter (Strategy Pattern).

The function must accept two tensors of shape `[N, C]` and return a `Cav` object.

In [ ]:
def normalized_centroid_cav(
    concept_features: torch.Tensor,
    random_features: torch.Tensor,
) -> Cav:
    """
    Custom CAV: centroid difference normalized to unit length.

    Normalizing ensures attribution scores are comparable across
    concepts with different activation magnitudes.
    """
    direction = (
        torch.mean(concept_features, dim=0)
        - torch.mean(random_features, dim=0)
    )
    direction = direction / (torch.norm(direction) + 1e-10)
    return Cav(direction=direction)


tcav_custom = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped", "dotted"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    cache_dir=CACHE_DIR,
    cav_fn=normalized_centroid_cav,
)

# cache_cav=False forces recomputation with the custom function
tcav_custom.explain(cache_cav=False, cache_random=True)
tcav_custom.plot(figsize=(12, 5))

### Default vs custom CAV — comparison

In [ ]:
import numpy as np

concepts = ["striped", "dotted"]
layer = "layer4"
top_class = tcav_local.target_classes[0]

default_scores = [
    tcav_local.computations[layer][c].attributions.get(
        top_class, torch.tensor(0.0)
    ).item()
    for c in concepts
]
custom_scores = [
    tcav_custom.computations[layer][c].attributions.get(
        top_class, torch.tensor(0.0)
    ).item()
    for c in concepts
]

x = np.arange(len(concepts))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, default_scores, width,
       label="Default (centroid diff)", color="steelblue")
ax.bar(x + width/2, custom_scores, width,
       label="Custom (normalized)", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(concepts)
ax.set_ylabel("Attribution score")
ax.set_title(
    f"Default vs Custom CAV — layer4 — "
    f"{tcav_local.model_wrapper.id_to_label(top_class)}"
)
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

---

## 7. GlobalVisualTCAV

Runs the pipeline on a folder of test images and answers:

**Does this concept consistently influence predictions for this class?**

In [ ]:
tcav_global = GlobalVisualTCAV(
    model="resnet50",
    test_images_dir=os.path.join(TEST_IMAGES_DIR, "zebra"),
    concept_names=["striped", "dotted", "zigzagged"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    n_classes=3,
    max_test_images=50,
    cache_dir=CACHE_DIR,
)

tcav_global.explain(cache_cav=True, cache_random=True)

In [ ]:
tcav_global.statsInfo()

In [ ]:
tcav_global.plot(figsize=(10, 5))

---

## 8. Saving results to disk

In [ ]:
os.makedirs("./results", exist_ok=True)

tcav_local.plot(
    figsize=(14, 10),
    save_path="./results/zebra_local_explanation.png",
)
tcav_global.plot(
    figsize=(10, 5),
    save_path="./results/zebra_global_explanation.png",
)

print("Figures saved to ./results/")

---

## 9. Interpreting the results

### Concept map
- **Red/yellow areas** — the CNN strongly detects the concept here
- **Black areas** — the concept is absent at that location
- Upscaled from layer spatial resolution (7×7 for ResNet50 layer4)
  to image size (224×224) using bilinear interpolation

### Attribution score
- Scores are **not percentages** — they are raw scalar values
- What matters is the **relative ranking** between concepts
- A much higher score for `striped` than `dotted` on a zebra means
  the model uses stripes more than dots for that classification

### Global confidence interval
| CI | Interpretation |
|---|---|
| Narrow (e.g. [0.18, 0.22]) | Concept consistently matters ✅ |
| Wide (e.g. [0.02, 0.38]) | Concept matters for some images but not others ⚠️ |
| Including 0 | Concept does not reliably influence predictions ❌ |

---

## 10. Text-to-Concept (optional — requires CLIP)

Generates CAVs from plain text instead of concept images.

```bash
pip install visual-tcav[text-to-concept]
```

In [ ]:
try:
    from visual_tcav import TextToConcept

    t2c = TextToConcept(model_wrapper=tcav_local.model_wrapper)
    t2c.load_aligner("./aligners/resnet50_layer4.pt")

    cav_from_text = t2c.get_cav_from_text("stripes", layer_name="layer4")
    print(f"CAV generated from text: {cav_from_text}")
    print(f"Direction shape: {cav_from_text.direction.shape}")

except ImportError:
    print("CLIP not installed. Run: pip install visual-tcav[text-to-concept]")
except FileNotFoundError:
    print("Linear Aligner not found. Train it first using train_aligners.py")

---

## Summary

| Feature | How to use |
|---|---|
| **Inspect layers** | `available_layers("resnet50")` |
| **Model as string** | `LocalVisualTCAV(model="resnet50", ...)` |
| **Model as nn.Module** | `LocalVisualTCAV(model=my_resnet, model_name="resnet50", ...)` |
| **Configurable paths** | Pass `concept_base_dir`, `random_dir`, `cache_dir` freely |
| **Skip predict()** | `explain()` calls it automatically |
| **Multiple layers** | `layer_names=["layer2", "layer3", "layer4"]` |
| **Custom CAV** | `LocalVisualTCAV(cav_fn=my_function, ...)` |
| **Global explanation** | `GlobalVisualTCAV` with `test_images_dir` |
| **Statistics** | `statsInfo()` — mean, std, 95% CI |
| **Save figures** | `plot(save_path="./output.png")` |
| **Text-to-Concept** | `TextToConcept` + `load_aligner()` + `get_cav_from_text()` |
| **CLI** | `visual-tcav local --model resnet50 --image ...` |

---

**GitHub:** https://github.com/saracavallini01/visual-tcav  
**PyPI:** `pip install visual-tcav`